In [ ]:
!pip install -q langchain langchain-community langchain-core
!pip install -q sentence-transformers faiss-cpu rank_bm25
!pip install -q pypdf gradio transformers accelerate
!pip install -q langchain-groq

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
from langchain.document_loaders import PyPDFLoader

def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print("📄 Loaded Documents (head):")
    print(documents[:2])
    return documents

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def chunk_docs(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documents)
    
    print("\n✂️ Chunks (head):")
    for c in chunks[:2]:
        print(c.page_content[:200])
    
    return chunks

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def tokenize_chunks(chunks):
    tokenized = [tokenizer.tokenize(c.page_content) for c in chunks]
    
    print("\n🔤 Tokenized Output (head):")
    print(tokenized[:2])
    
    return tokenized

In [ ]:
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer('all-MiniLM-L6-v2')

def dense_embed(chunks):
    texts = [c.page_content for c in chunks]
    embeddings = dense_model.encode(texts)
    
    print("\n🧬 Dense Embeddings Shape:", embeddings.shape)
    print("Head:", embeddings[:2])
    
    return embeddings, texts

In [ ]:
from rank_bm25 import BM25Okapi

def sparse_embed(tokenized_chunks):
    bm25 = BM25Okapi(tokenized_chunks)
    
    print("\n📊 BM25 Ready (corpus size):", len(tokenized_chunks))
    
    return bm25

In [ ]:
import faiss
import numpy as np

def build_faiss(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    
    print("\n🗂️ FAISS Index Built")
    print("Total vectors:", index.ntotal)
    
    return index

In [ ]:
def hybrid_retrieve(query, dense_model, index, texts, bm25, tokenizer, top_k=3):
    # Dense retrieval
    q_embed = dense_model.encode([query])
    D, I = index.search(np.array(q_embed), top_k)
    
    dense_results = [texts[i] for i in I[0]]
    
    # Sparse retrieval
    tokenized_query = tokenizer.tokenize(query)
    sparse_scores = bm25.get_scores(tokenized_query)
    top_sparse_idx = np.argsort(sparse_scores)[::-1][:top_k]
    sparse_results = [texts[i] for i in top_sparse_idx]
    
    print("\n🔍 Dense Results (head):")
    print(dense_results[:1])
    
    print("\n🔍 Sparse Results (head):")
    print(sparse_results[:1])
    
    # Combine
    combined = list(set(dense_results + sparse_results))
    
    return combined

In [ ]:
def generate_answer(context, query):
    prompt = f"""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "Not found in document."

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    print("\n🤖 LLM Output (head):")
    print(response.content[:300])

    return response.content

In [ ]:
def summarize(texts):
    combined = " ".join(texts[:5])
    
    prompt = f"""
Summarize the following document clearly:

{combined}
"""
    response = llm.invoke(prompt)
    
    return response.content

In [ ]:
import gradio as gr

def process_pdf(file, query):
    documents = load_pdf(file.name)
    chunks = chunk_docs(documents)
    tokenized = tokenize_chunks(chunks)
    
    embeddings, texts = dense_embed(chunks)
    bm25 = sparse_embed(tokenized)
    index = build_faiss(embeddings)
    
    if query.strip() == "":
        return summarize(texts)
    
    retrieved = hybrid_retrieve(query, dense_model, index, texts, bm25, tokenizer)
    context = "\n".join(retrieved)
    
    return generate_answer(context, query)


with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 📚 Hybrid RAG PDF Assistant (Groq Powered)")
    
    file_input = gr.File(label="Upload PDF")
    query_input = gr.Textbox(label="Ask a question (leave empty for summary)")
    output = gr.Textbox(label="Output", lines=15)
    
    btn = gr.Button("Run")
    
    btn.click(
        process_pdf,
        inputs=[file_input, query_input],
        outputs=output
    )

app.launch(debug=True)